
## Functions in PySpark:


Functions in PySpark are `predefined or user-defined operations` used to `transform, process, clean, and analyze data` in a `distributed environment`.


- **Built-in Functions**
- **UDF (User Defined Function)**
- **Pandas UDF**


### 1. Built-in Functions:

A **Built-in functions** are `predefined operations` used to perform `transform, process, clean, and analyze data` without `writing custom Python code`.

## Common Built-in Functions in PySpark

| Function | Purpose | Example |
|---|---|---|
| `col()` | Access a column | `col("salary")` |
| `upper()` | Convert text to uppercase | `upper(col("name"))` |
| `lower()` | Convert text to lowercase | `lower(col("name"))` |
| `when()` | Apply conditional logic | `when(col("salary") > 50000, "High")` |
| `concat()` | Combine multiple columns | `concat(col("first_name"), col("last_name"))` |
| `sum()` | Calculate the total | `sum(col("salary"))` |
| `count()` | Count records | `count(col("EmployeeID"))` |
| `avg()` | Calculate the average | `avg(col("salary"))` |
| `to_date()` | Convert a string or timestamp to a date | `to_date(col("hiredate"), "yyyy-MM-dd")` |


### Real-Time Data Engineering Examples

| Function | Real-Time Use Case |
|---|---|
| `col()` | Select employee salary from an employee dataset |
| `upper()` | Standardize customer names to uppercase |
| `lower()` | Convert email addresses to lowercase |
| `when()` | Categorize employees based on salary |
| `concat()` | Combine first name and last name |
| `sum()` | Calculate total sales revenue |
| `count()` | Count total orders in an order dataset |
| `avg()` | Calculate average transaction value |
| `to_date()` | Convert incoming date strings into proper date values |


In [0]:
### Example Program

from pyspark.sql.functions import (
    col, upper, lower, when, concat,
    sum, count, avg, to_date
)

data = [
    (1, "Franklin", "Surya", 60000, "2024-01-15"),
    (2, "Arun", "Kumar", 40000, "2023-06-20"),
    (3, "Priya", "Devi", 70000, "2022-03-10")
]

columns = ["EmployeeID", "FirstName", "LastName", "Salary", "Hiredate"]

df = spark.createDataFrame(data, columns)

# 1. col() - Access a column
df.select(col("FirstName")).show()

# 2. upper() - Convert to uppercase
df.withColumn("UpperName", upper(col("FirstName"))).show()

# 3. lower() - Convert to lowercase
df.withColumn("LowerName", lower(col("FirstName"))).show()

# 4. when() - Apply conditions
df.withColumn(
    "SalaryCategory",
    when(col("Salary") >= 50000, "High").otherwise("Low")
).show()

# 5. concat() - Combine columns
df.withColumn(
    "FullName",
    concat(col("FirstName"), col("LastName"))
).show()

# 6. sum() - Calculate total salary
df.select(sum(col("Salary")).alias("TotalSalary")).show()

# 7. count() - Count employees
df.select(count(col("EmployeeID")).alias("TotalEmployees")).show()

# 8. avg() - Calculate average salary
df.select(avg(col("Salary")).alias("AverageSalary")).show()

# 9. to_date() - Convert string to date
df.withColumn(
    "HireDateFormatted",
    to_date(col("Hiredate"), "yyyy-MM-dd")
).show()


**Real-time example: Standardize employee names.**

In [0]:
from pyspark.sql.functions import *

df = df.withColumn(
    "FirstName",
    upper(trim(col("FirstName")))
)

df.show()


### 2. UDF (User Defined Function):

- A **User Defined Function** is `defined by user` using `Python function` and use it in `PySpark` when built-in functions are `insufficient`.

- We can define the **User Defined Function** using `udf()`.


**Real-time example: Calculate employee bonus.**

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

# Create Python function
def calculate_bonus(salary):
    return salary * 0.10

# Register as UDF
bonus_udf = udf(calculate_bonus, DoubleType())

# Apply UDF
df = df.withColumn(
    "Bonus",
    bonus_udf(col("Salary"))
)

df.show()


**Create a temporary view**

In [0]:
df.createOrReplaceTempView("employees")


## Register UDF for SQL.

In [0]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import DoubleType

# 1. Define the Python function
def calculate_bonus(salary):
    if salary is None:
        return 0.0

    if salary >= 40000:
        return float(salary * 0.10)

    return 0.0


# 2. Register the function for Spark SQL
spark.udf.register(
    "calculate_bonus",
    calculate_bonus,
    DoubleType()
)

print("UDF registered successfully!")


### Use in Spark SQL:

In [0]:
%sql
SELECT
    EmployeeID,
    FirstName,
    Salary,
    calculate_bonus(Salary) AS Bonus
FROM employees;


### 3. Pandas UDF:

**Pandas UDF** is `a user-defined function` which uses `Pandas and Apache Arrow` to **process data in batches** instead of **processing individual rows**.


**Real-time example: Calculate a 10% bonus using Pandas UDF.**

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf("double")
def calculate_bonus(salary: pd.Series) -> pd.Series:
    return salary * 0.10

df = df.withColumn(
    "Bonus",
    calculate_bonus(col("Salary"))
)


### 4. Why should we avoid UDFs?

- We should avoid UDFs because `they introduce Python serialization and execution overhead`.

- UDFs can `reduce Spark pipeline performance`.

### 5. Reusable Function Library:

It is a `collection of common functions that can be imported and reused` across` multiple ETL pipelines`.

**Example: Create a Python file called common_functions.py**

In [0]:
from common_functions import standardize_column

df = standardize_column(df, "FirstName")

df.show()


### 6. Generic Framework Functions:

Generic Framework Functions are `import and reusable functions` designed to handle` common data engineering operations`.

## Reusable / Generic Framework Functions in PySpark:

| Function | Purpose | Example Use |
|---|---|---|
| `read_file()` | Read CSV, JSON, and Parquet files | Load employee data |
| `write_file()` | Write data into files | Save processed data as Parquet |
| `write_table()` | Save data into database or Delta tables | Store transformed employee records |
| `standardize_column()` | Standardize column values | Convert names to uppercase, remove extra spaces |
| `standardize_schema()` | Maintain consistent column names and data types | Rename columns and apply correct data types |

### Real-Time Data Engineering Use Cases:

| Function | Real-Time Example |
|---|---|
| `read_file()` | Read daily sales data from cloud storage |
| `write_file()` | Export cleaned data into Parquet format |
| `write_table()` | Store transformed data in a Delta table |
| `standardize_column()` | Remove extra spaces and standardize customer names |
| `standardize_schema()` | Ensure multiple source files have the same schema |

### Important interview questions:


### 1. Which is faster: Built-in functions or UDFs?

Built-in functions are `faster` because `Spark can optimize them using Catalyst`.

### 2. What is the main difference between UDF and Pandas UDF?

Regular UDF processes `individual rows`, whereas Pandas UDF processes `batches using Pandas and Apache Arrow`.

### 3. Can we use UDF in Spark SQL?

- Yes. 
- We can register a UDF using `spark.udf.register()` and call it `inside SQL queries`.

### 4. Why should we avoid Python UDFs?

- We should avoid UDFs because they introduce `Python serialization and execution overhead`.
- UDFs can `reduce Spark pipeline performance`.

### 5. What is Apache Arrow?

Apache Arrow is an `in-memory columnar data forma`t that enables `efficient data exchange between JVM and Python`, especially for `Pandas UDFs`.

### 6. What is a reusable function library?

It is a `collection of common functions that can be imported and reused` across `multiple ETL pipelines`.